# 01 — MNIST 数据探索

看清楚我们要分类的是什么：像素分布、类别平衡、几个样本长什么样。
也确认下我们的 `[-1, 1]` 归一化对不对。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from backend.data import load_mnist_sklearn

X_tr, y_tr, X_te, y_te = load_mnist_sklearn()
print(f'train: {X_tr.shape}, {y_tr.shape}')
print(f'test : {X_te.shape}, {y_te.shape}')
print(f'pixel range: [{X_tr.min():.2f}, {X_tr.max():.2f}]')
print(f'pixel dtype: {X_tr.dtype}')

## 类别分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, y, name in [(axes[0], y_tr, 'train'), (axes[1], y_te, 'test')]:
    counts = np.bincount(y, minlength=10)
    ax.bar(range(10), counts, color='#38bdf8')
    ax.set_title(f'{name} class counts')
    ax.set_xticks(range(10))
plt.tight_layout(); plt.show()

## 随机看 5 个样本（每行一个 class）
确认归一化看起来正常：背景应该是深色（-1），笔迹是浅色（+1）。

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(10, 5, figsize=(8, 16))
for cls in range(10):
    idx = np.where(y_tr == cls)[0]
    picks = rng.choice(idx, 5, replace=False)
    for j, k in enumerate(picks):
        axes[cls, j].imshow(X_tr[k, 0], cmap='gray', vmin=-1, vmax=1)
        axes[cls, j].axis('off')
        if j == 0:
            axes[cls, j].set_ylabel(str(cls), rotation=0, fontsize=14, labelpad=15)
plt.suptitle('5 random samples per class', y=1.005)
plt.tight_layout(); plt.show()

## 像素值分布
MNIST 大多数像素是背景（接近 -1），少数是笔迹（接近 +1）。
这张图能让你直观感受到为啥背景填充值要选 -1（不然 padding 会引入虚假的 "笔迹"）。

In [ ]:
plt.figure(figsize=(6, 3))
plt.hist(X_tr[:1000].ravel(), bins=50, color='#34d399', alpha=0.8)
plt.yscale('log')
plt.xlabel('pixel value')
plt.ylabel('count (log)')
plt.title('Pixel value distribution (1000 samples)')
plt.tight_layout(); plt.show()

## Pad 到 32×32 看看
LeNet-5 的标准输入是 32×32，让 5×5 conv 在第一层就能看到边缘像素。

In [ ]:
from backend.data import pad_to_32

x_padded = pad_to_32(X_tr[:1])
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(X_tr[0, 0], cmap='gray', vmin=-1, vmax=1)
axes[0].set_title(f'original 28×28, label={y_tr[0]}')
axes[1].imshow(x_padded[0, 0], cmap='gray', vmin=-1, vmax=1)
axes[1].set_title('padded 32×32 (with -1)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()